In [155]:
# Importy
import os, json, pickle, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pretty_midi
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
#from tqdm.notebook import tqdm 

from functions import DoomDataset

In [156]:
# Parametry
VOCAB_SIZE = 579
EMBEDDING_DIM = 256
HIDDEN_SIZE = 256
NUM_LAYERS = 2
STRIDE = 16 
CONTEXT = 256
BATCH_SIZE = 256
EPOCHS = 30
LR = 0.001
MAX_TRANSPOSE = 12
STEPS_PER_BEAT = 8
# Zmienic przed odpaleniem teningu. 
# Wszyscy wiemy, że zajmująca maks 5 minut implementacja prostego sprawdzania, czy plik istnieje jest zbędna i niepotrzebnie komplikuje kod prawda?
FILE_NAME_SAVE = 'lstm_music_model_30ep_2Layers_256hidden_lr0001_context256_ed256_stride16_bs256_V12.pth'
FILE_NAME_LOAD = 'lstm_music_model_30ep_2Layers_256hidden_lr0001_context256_ed256_stride16_bs256_V12.pth'

#lstm_music_model_50ep_2Layers_512hidden_lr0005_context512_ed512_stride256_V7.pth
# best lstm_music_model_60ep_2Layers_512hidden_lr0005_context512_ed512_stride256_V9
os.makedirs('../models', exist_ok=True)
os.makedirs('../output', exist_ok=True)
os.makedirs('../output/LSTM', exist_ok=True)

PROC_DIR = '../data/processed/'
OUTPUT_DIR = '../output/LSTM/'
MODEL_DIR = '../models/'

In [157]:
# Ustwaianie ziarna i cuda
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Device: NVIDIA GeForce RTX 4060 Laptop GPU


In [158]:
# Ładowanie danych i słownika tokenów
with open(os.path.join(PROC_DIR, 'sequences.pkl'), 'rb') as f:
    data = pickle.load(f)

with open(os.path.join(PROC_DIR, 'vocab.json')) as f:
    token2id = json.load(f)
id2token = {y: x for x,y in token2id.items()}

sequences = data.values()
names = list(data.keys())
VOCAB_SIZE = len(token2id)
PAD_ID = token2id['PAD']
print(f'Sequences: {len(sequences)} | Vocab: {VOCAB_SIZE} | Tokens: {sum(len(s) for s in sequences):,}')

Sequences: 43 | Vocab: 579 | Tokens: 384,327


In [159]:
# Przygotowanie DataLoaderów
sequences_list = list(sequences)
random.shuffle(sequences_list)
train_sequences, val_sequences = sequences_list[:int(0.8*len(sequences_list))], sequences_list[int(0.8*len(sequences_list)):]
train_dataset = DoomDataset(train_sequences, context = CONTEXT, stride = STRIDE, augment=True, max_transpose=MAX_TRANSPOSE)
print(f"Total training samples generated: {len(train_dataset)}")
print(f"With batch_size={BATCH_SIZE}, one epoch will take {len(train_dataset) // BATCH_SIZE} steps.")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = DoomDataset(val_sequences, context = CONTEXT, stride=STRIDE, augment=False, max_transpose=MAX_TRANSPOSE)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

Total training samples generated: 18736
With batch_size=256, one epoch will take 73 steps.


In [160]:
# Pobranie jednej paczki danych
features, labels = next(iter(train_loader))

# Sprawdzenie kształtu danych
print(f"Kształt cech (X): {features.shape}")
print(f"Kształt etykiet (y): {labels.shape}")


Kształt cech (X): torch.Size([256, 256])
Kształt etykiet (y): torch.Size([256, 256])


In [161]:
# Definicja modelu
class MusicLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super(MusicLSTM, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, batch_first=True, dropout=0.6)
        
        self.ln = nn.LayerNorm(hidden_size)

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, hidden=None):
        
        embedded = self.embedding(x)
        
        out, hidden = self.lstm(embedded, hidden)
        
        out = self.ln(out)

        out = self.dropout(out)

        logits = self.fc(out)
        
        return logits, hidden
  

In [162]:
# Inicjalizacja modelu, optymalizatora i funkcji straty
model = MusicLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS).to(device)

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)


In [163]:
# Trenowanie modelu
import torch
from torch.optim.lr_scheduler import StepLR

scheduler = StepLR(optimizer, step_size=10, gamma=0.5)

for epoch in range(EPOCHS):
   
    model.train()
    total_loss = 0
    total_batches = len(train_loader)

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()

        logits, _ = model(inputs)

        loss = loss_fn(logits.view(-1, VOCAB_SIZE), targets.view(-1))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 40 == 0 or batch_idx == total_batches - 1:
            current_avg_loss = total_loss / (batch_idx + 1)
            print(f"Epoch: [{epoch+1}/{EPOCHS}] | Batch: [{batch_idx + 1}/{total_batches}] | Train Loss: {current_avg_loss:.4f}")

    epoch_train_loss = total_loss / total_batches
    print(f"Epoch {epoch+1} Train Summary: Avg Loss = {epoch_train_loss:.4f} ")

    model.eval() 
    total_val_loss = 0
    val_batches = len(val_loader)

    with torch.no_grad():
        for val_inputs, val_targets in val_loader:
            val_inputs = val_inputs.to(device)
            val_targets = val_targets.to(device)

            val_logits, _ = model(val_inputs)
            val_loss = loss_fn(val_logits.view(-1, VOCAB_SIZE), val_targets.view(-1))
            
            total_val_loss += val_loss.item()

    epoch_val_loss = total_val_loss / val_batches
    print(f"Epoch {epoch+1} Val Summary: Avg Loss = {epoch_val_loss:.4f}")

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Learning rate for next epoch: {current_lr}\n")


Epoch: [1/30] | Batch: [1/74] | Train Loss: 6.6692
Epoch: [1/30] | Batch: [41/74] | Train Loss: 4.7907
Epoch: [1/30] | Batch: [74/74] | Train Loss: 4.2723
Epoch 1 Train Summary: Avg Loss = 4.2723 
Epoch 1 Val Summary: Avg Loss = 4.0939
Learning rate for next epoch: 0.001

Epoch: [2/30] | Batch: [1/74] | Train Loss: 3.3845
Epoch: [2/30] | Batch: [41/74] | Train Loss: 3.1881
Epoch: [2/30] | Batch: [74/74] | Train Loss: 3.0825
Epoch 2 Train Summary: Avg Loss = 3.0825 
Epoch 2 Val Summary: Avg Loss = 3.8704
Learning rate for next epoch: 0.001

Epoch: [3/30] | Batch: [1/74] | Train Loss: 2.9067
Epoch: [3/30] | Batch: [41/74] | Train Loss: 2.8246
Epoch: [3/30] | Batch: [74/74] | Train Loss: 2.7754
Epoch 3 Train Summary: Avg Loss = 2.7754 
Epoch 3 Val Summary: Avg Loss = 3.8312
Learning rate for next epoch: 0.001

Epoch: [4/30] | Batch: [1/74] | Train Loss: 2.6662
Epoch: [4/30] | Batch: [41/74] | Train Loss: 2.6439
Epoch: [4/30] | Batch: [74/74] | Train Loss: 2.6146
Epoch 4 Train Summary: Avg

In [164]:
# Zapis modelu
torch.save(model.state_dict(), os.path.join(MODEL_DIR, FILE_NAME_SAVE))

In [165]:
# Generacja muzyki
def generate_music(model, token2id, id2token, max_generated_tokens=2000, temperature=1.1):
    model.eval()
    
    generated = [token2id['BOS']]
    
    hidden = None
    
    with torch.no_grad():
        for _ in range(max_generated_tokens):

            input_tensor = torch.tensor([[generated[-1]]], dtype=torch.long).to(device)
            
            logits, hidden = model(input_tensor, hidden)
            
            logits = logits[0, -1, :]
            
            logits = logits / temperature
            
            probabilities = F.softmax(logits, dim=-1)
            
            next_token_id = torch.multinomial(probabilities, num_samples=1).item()
            
            generated.append(next_token_id)
            
            if next_token_id == token2id.get('EOS'):
                break
                
    generated_tokens = [id2token[idx] for idx in generated]
    return generated_tokens




In [166]:
# Generowanie utworów
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, FILE_NAME_LOAD)))

generated_sequence = generate_music(model, token2id, id2token, max_generated_tokens=2700, temperature=0.8)
print(f"Generated {len(generated_sequence)} tokens.")
print("Example fragment::", generated_sequence[:30])

Generated 2701 tokens.
Example fragment:: ['BOS', 'PROGRAM_30', 'NOTE_ON_43', 'PROGRAM_30', 'NOTE_ON_50', 'PROGRAM_33', 'NOTE_ON_31', 'DRUM_42', 'DRUM_55', 'DRUM_36', 'SHIFT_2', 'NOTE_OFF_43', 'NOTE_OFF_50', 'NOTE_OFF_31', 'SHIFT_1', 'PROGRAM_30', 'NOTE_ON_45', 'PROGRAM_30', 'NOTE_ON_52', 'PROGRAM_33', 'NOTE_ON_33', 'DRUM_36', 'SHIFT_2', 'NOTE_OFF_45', 'NOTE_OFF_52', 'NOTE_OFF_33', 'SHIFT_1', 'PROGRAM_30', 'NOTE_ON_48', 'PROGRAM_30']


In [167]:
# Konwersja tokenów do MIDI
def events_to_midi(events, bpm=95):
    sec_per_step = (60.0 / bpm) / STEPS_PER_BEAT
    midi      = pretty_midi.PrettyMIDI()
    insts     = {}     # program -> Instrument (melodyczne)
    drum_inst = None   # jeden wspólny kanał perkusji
    active    = {}     # pitch -> [(start_step, program)]
    cur, last_prog = 0, 0

    for ev in events:
        if ev in ('BOS', 'EOS', 'PAD'):
            continue
        if ev.startswith('SHIFT_'):
            cur += int(ev.split('_')[1])
        elif ev.startswith('PROGRAM_'):
            last_prog = int(ev.split('_')[1])
        elif ev.startswith('NOTE_ON_'):
            pitch = int(ev.split('_')[-1])
            active.setdefault(pitch, []).append((cur, last_prog))
        elif ev.startswith('NOTE_OFF_'):
            pitch = int(ev.split('_')[-1])
            if active.get(pitch):
                start_step, prog = active[pitch].pop(0)
                start = start_step * sec_per_step
                end   = cur * sec_per_step
                if end > start:
                    if prog not in insts:
                        insts[prog] = pretty_midi.Instrument(program=prog)
                    insts[prog].notes.append(pretty_midi.Note(100, pitch, start, end))
        elif ev.startswith('DRUM_'):
            pitch = int(ev.split('_')[1])
            start = cur * sec_per_step
            end   = start + sec_per_step
            if drum_inst is None:
                drum_inst = pretty_midi.Instrument(program=0, is_drum=True)
            drum_inst.notes.append(pretty_midi.Note(100, pitch, start, end))

    for inst in insts.values():
        midi.instruments.append(inst)
    if drum_inst is not None:
        midi.instruments.append(drum_inst)
    return midi

In [168]:
# Zapis wygenerowanej muzyki do pliku MIDI
events_to_midi(generated_sequence, bpm = 90).write(os.path.join(OUTPUT_DIR, "Generated_DOOM_V21.mid"))